# Re-optimización de XGBoost con Optuna para configuraciones demográficas

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "src").exists():
    ROOT = _cwd
elif (_cwd.parent / "src").exists():
    ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_cwd}")

REPO = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Code root:", ROOT)
print("Repo root:", REPO)

Code root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Code
Repo root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
import optuna

from src.config import Paths, ExperimentConfig
from src.cohort import build_final_cohort_df
from src.data_io import read_full_metadata
from src.splits import load_splits
from src.utils_ids import normalize_record_id
from src.figures import readable_feature_name

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)
cfg = ExperimentConfig(seed=42)
np.random.seed(cfg.seed)

IMGDIR = REPO / "Latex" / "06_resultados_discusion" / "images"
IMGDIR.mkdir(parents=True, exist_ok=True)
OUTDIR = paths.out_dir / "experiments" / "demographics_optuna"
OUTDIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})

/home/nico/Desktop/5to/TesisFinal/.tesis-final-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-26 16:03:40.910758: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-26 16:03:41.435504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 16:03:42.526664: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
cohort = build_final_cohort_df(
    excel_path=paths.excel_path,
    fc_folder=paths.fc_folder,
    t1w_csv_path=paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)

splits = load_splits(paths.out_dir / "splits" / "splits_seed42_test0.1.json")
trainval_ids = splits["holdout"]["trainval_ids"]
test_ids = splits["holdout"]["test_ids"]

Z_trainval = np.load(paths.out_dir / "embeddings" / "mu_trainval.npy")
Z_test = np.load(paths.out_dir / "embeddings" / "mu_test.npy")

with open(paths.out_dir / "embeddings" / "mu_trainval.json") as f:
    emb_tv_meta = json.load(f)
with open(paths.out_dir / "embeddings" / "mu_test.json") as f:
    emb_te_meta = json.load(f)

emb_tv_ids = emb_tv_meta["ids"]
emb_te_ids = emb_te_meta["ids"]

full_meta = read_full_metadata(paths.excel_path)
demo_cols = ["record_id", "site", "cog_ed", "mmse_total", "mri_scanner_brand"]
demo = full_meta[demo_cols].copy()
merged = cohort[["record_id", "age", "sex", "diagnosis"]].merge(demo, on="record_id", how="left")

print(f"Cohort: {len(cohort)}, Trainval: {len(trainval_ids)}, Test: {len(test_ids)}")

Cohort: 1245, Trainval: 1120, Test: 125


In [4]:
t1_cols = [c for c in cohort.columns if c.startswith("t1_")]

def build_demo_features(ids_list, Z_array, emb_ids, cohort_df, merged_df,
                        use_education=False, use_site=False):
    cohort_idx = {r: i for i, r in enumerate(cohort_df["record_id"].values)}
    emb_idx = {r: i for i, r in enumerate(emb_ids)}
    merged_idx = {r: i for i, r in enumerate(merged_df["record_id"].values)}

    valid_ids = []
    for rid in ids_list:
        if rid not in cohort_idx or rid not in emb_idx or rid not in merged_idx:
            continue
        row_m = merged_df.iloc[merged_idx[rid]]
        if use_education and pd.isna(row_m.get("cog_ed")):
            continue
        if use_site and pd.isna(row_m.get("site")):
            continue
        valid_ids.append(rid)

    c_rows = [cohort_idx[r] for r in valid_ids]
    e_rows = [emb_idx[r] for r in valid_ids]
    m_rows = [merged_idx[r] for r in valid_ids]

    c_sub = cohort_df.iloc[c_rows]
    Z = Z_array[e_rows]
    T1 = c_sub[t1_cols].values.astype(np.float32)
    y = c_sub["age"].values.astype(np.float64)

    parts = [Z, T1]
    feat_names = [f"z_{i}" for i in range(Z.shape[1])] + list(t1_cols)

    if use_education:
        ed = merged_df.iloc[m_rows]["cog_ed"].values.astype(np.float32).reshape(-1, 1)
        parts.append(ed)
        feat_names.append("education")

    if use_site:
        site_vals = merged_df.iloc[m_rows]["site"]
        site_dum = pd.get_dummies(site_vals, prefix="site")
        parts.append(site_dum.values.astype(np.float32))
        feat_names.extend(list(site_dum.columns))

    X = np.concatenate(parts, axis=1).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, y, valid_ids, feat_names

## Optuna re-optimization for each configuration

In [5]:
configs = [
    {"name": "VAE + T1w + educación",         "use_education": True,  "use_site": False},
    {"name": "VAE + T1w + sitio",             "use_education": False, "use_site": True},
    {"name": "VAE + T1w + educación + sitio", "use_education": True,  "use_site": True},
]

N_TRIALS = 100
K_FOLDS = 5

all_results = []

for c in configs:
    print(f"\n{'='*60}")
    print(f"Optimizing: {c['name']}")
    print(f"{'='*60}")

    X_tr, y_tr, ids_tr, fnames_tr = build_demo_features(
        trainval_ids, Z_trainval, emb_tv_ids, cohort, merged,
        use_education=c["use_education"], use_site=c["use_site"]
    )
    X_te, y_te, ids_te, fnames_te = build_demo_features(
        test_ids, Z_test, emb_te_ids, cohort, merged,
        use_education=c["use_education"], use_site=c["use_site"]
    )

    print(f"  Train: {X_tr.shape}, Test: {X_te.shape}")

    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=cfg.seed)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 2500),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 10.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "tree_method": "hist",
            "random_state": cfg.seed,
            "eval_metric": "mae",
            "verbosity": 0,
        }
        maes = []
        for tr_idx, va_idx in kf.split(X_tr):
            model = XGBRegressor(**params)
            model.fit(X_tr[tr_idx], y_tr[tr_idx], verbose=False)
            maes.append(mean_absolute_error(y_tr[va_idx], model.predict(X_tr[va_idx])))
        return float(np.mean(maes))

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=cfg.seed))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_trial.params
    best_params.update({"tree_method": "hist", "random_state": cfg.seed,
                        "eval_metric": "mae", "verbosity": 0})

    model_final = XGBRegressor(**best_params)
    model_final.fit(X_tr, y_tr, verbose=False)
    pred_te = model_final.predict(X_te)

    mae = mean_absolute_error(y_te, pred_te)
    r2 = r2_score(y_te, pred_te)
    pearson_r = float(np.corrcoef(y_te, pred_te)[0, 1])

    result = {
        "config": c["name"],
        "n_train": len(X_tr), "n_test": len(X_te),
        "n_features": X_tr.shape[1],
        "cv_mae": study.best_value,
        "test_mae": mae, "test_r2": r2, "test_pearson": pearson_r,
        "best_params": best_params,
        "feature_names": fnames_te,
    }
    all_results.append(result)

    config_slug = c["name"].replace(" ", "_").replace("+", "").replace("á", "a").replace("ó", "o")
    with open(OUTDIR / f"{config_slug}.json", "w") as f:
        json.dump(result, f, indent=2, default=float)

    print(f"\n  Best CV MAE: {study.best_value:.2f}")
    print(f"  Test MAE: {mae:.2f}, R²: {r2:.3f}, Pearson: {pearson_r:.3f}")
    print(f"  Best params: { {k: round(v,4) if isinstance(v,float) else v for k,v in best_params.items()} }")


Optimizing: VAE + T1w + educación
  Train: (842, 181), Test: (91, 181)


Best trial: 81. Best value: 5.57717: 100%|██████████| 100/100 [25:54<00:00, 15.55s/it]



  Best CV MAE: 5.58
  Test MAE: 5.33, R²: 0.502, Pearson: 0.711
  Best params: {'n_estimators': 2302, 'max_depth': 4, 'learning_rate': 0.0093, 'subsample': 0.5, 'colsample_bytree': 0.5556, 'reg_alpha': 0.2902, 'reg_lambda': 0.0, 'min_child_weight': 0.1367, 'gamma': 0.4036, 'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mae', 'verbosity': 0}

Optimizing: VAE + T1w + sitio
  Train: (848, 189), Test: (93, 189)


Best trial: 97. Best value: 5.53583: 100%|██████████| 100/100 [25:54<00:00, 15.55s/it]



  Best CV MAE: 5.54
  Test MAE: 5.37, R²: 0.538, Pearson: 0.744
  Best params: {'n_estimators': 2049, 'max_depth': 4, 'learning_rate': 0.0121, 'subsample': 0.5316, 'colsample_bytree': 0.5638, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_weight': 0.721, 'gamma': 2.355, 'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mae', 'verbosity': 0}

Optimizing: VAE + T1w + educación + sitio
  Train: (842, 190), Test: (91, 190)


Best trial: 87. Best value: 5.54434: 100%|██████████| 100/100 [24:42<00:00, 14.83s/it]



  Best CV MAE: 5.54
  Test MAE: 5.17, R²: 0.523, Pearson: 0.725
  Best params: {'n_estimators': 2267, 'max_depth': 3, 'learning_rate': 0.0135, 'subsample': 0.5533, 'colsample_bytree': 0.5678, 'reg_alpha': 0.5994, 'reg_lambda': 0.0001, 'min_child_weight': 0.3037, 'gamma': 2.5982, 'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mae', 'verbosity': 0}


## Feature importance analysis

In [8]:
for result in all_results:
    c_name = result["config"]

    use_ed = "educación" in c_name or "educacion" in c_name
    use_site = "sitio" in c_name

    X_tr, y_tr, _, fnames = build_demo_features(
        trainval_ids, Z_trainval, emb_tv_ids, cohort, merged,
        use_education=use_ed, use_site=use_site
    )

    model = XGBRegressor(**result["best_params"])
    model.fit(X_tr, y_tr, verbose=False)

    importances = model.feature_importances_
    feat_imp = sorted(zip(fnames, importances), key=lambda x: x[1], reverse=True)

    top20 = feat_imp[:20]
    names, vals = zip(*top20)
    display_names = [readable_feature_name(n) for n in names]

    fig, ax = plt.subplots(figsize=(8, 6))
    y_pos = np.arange(len(top20))
    bars = ax.barh(y_pos, vals, align="center")

    for i, (name, _) in enumerate(top20):
        if name == "education":
            bars[i].set_color("#EF5350")
        elif name.startswith("site_"):
            bars[i].set_color("#FFB74D")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(display_names)
    ax.invert_yaxis()
    ax.set_xlabel("Gain")
    ax.set_title(f"Top-20 Feature Importance: {c_name}")
    plt.tight_layout()

    slug = c_name.replace(" ", "_").replace("+", "").replace("á", "a").replace("ó", "o")
    plt.savefig(IMGDIR / f"demo_importance_{slug}.png", bbox_inches="tight", dpi=300)
    plt.close()

    demo_feats = [(n, v) for n, v in feat_imp if n == "education" or n.startswith("site_")]
    if demo_feats:
        print(f"\n{c_name} — demographic feature ranks:")
        for n, v in demo_feats:
            rank = [x[0] for x in feat_imp].index(n) + 1
            print(f"  {n}: rank={rank}/{len(feat_imp)}, gain={v:.4f}")

print("\nDone. Figures saved to:", IMGDIR)


VAE + T1w + educación — demographic feature ranks:
  education: rank=29/181, gain=0.0077

VAE + T1w + sitio — demographic feature ranks:
  site_Miller: rank=1/189, gain=0.0382
  site_Lopera: rank=5/189, gain=0.0229
  site_Takada: rank=9/189, gain=0.0209
  site_Slachevsky: rank=51/189, gain=0.0053
  site_Matallana: rank=138/189, gain=0.0026
  site_Bruno: rank=160/189, gain=0.0022
  site_Behrens: rank=187/189, gain=0.0007
  site_Avila: rank=188/189, gain=0.0000
  site_Resende: rank=189/189, gain=0.0000

VAE + T1w + educación + sitio — demographic feature ranks:
  site_Miller: rank=2/190, gain=0.0428
  site_Lopera: rank=5/190, gain=0.0222
  site_Takada: rank=15/190, gain=0.0125
  site_Bruno: rank=22/190, gain=0.0092
  education: rank=27/190, gain=0.0075
  site_Matallana: rank=46/190, gain=0.0055
  site_Slachevsky: rank=152/190, gain=0.0022
  site_Behrens: rank=187/190, gain=0.0013
  site_Avila: rank=189/190, gain=0.0000
  site_Resende: rank=190/190, gain=0.0000

Done. Figures saved to: /

## Summary comparison

In [9]:
print("=" * 80)
print("COMPARISON: Original params vs. Re-optimized")
print("=" * 80)
print()

original = json.load(open(paths.out_dir / "experiments" / "demographics_analysis.json"))
orig_ablations = {r["config"]: r for r in original["ablations"]}

for result in all_results:
    name = result["config"]
    orig = orig_ablations.get(name, {})
    print(f"{name}:")
    if orig:
        print(f"  Original:     MAE={orig.get('MAE', 'N/A'):.2f}, R²={orig.get('R2', 'N/A'):.3f}")
    print(f"  Re-optimized: MAE={result['test_mae']:.2f}, R²={result['test_r2']:.3f}")
    if orig:
        delta = result["test_mae"] - orig.get("MAE", result["test_mae"])
        print(f"  Delta MAE:    {delta:+.2f}")
    print()

summary_path = OUTDIR / "summary.json"
with open(summary_path, "w") as f:
    json.dump(all_results, f, indent=2, default=float)
print("Full results saved to:", summary_path)

COMPARISON: Original params vs. Re-optimized

VAE + T1w + educación:
  Original:     MAE=5.26, R²=0.504
  Re-optimized: MAE=5.33, R²=0.502
  Delta MAE:    +0.07

VAE + T1w + sitio:
  Original:     MAE=5.51, R²=0.534
  Re-optimized: MAE=5.37, R²=0.538
  Delta MAE:    -0.14

VAE + T1w + educación + sitio:
  Original:     MAE=5.37, R²=0.483
  Re-optimized: MAE=5.17, R²=0.523
  Delta MAE:    -0.20

Full results saved to: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/experiments/demographics_optuna/summary.json
